# 05 - Gold Summary Refresh

Sanitized portfolio notebook for the Corporate Travel Booking Data Platform.

Before running this notebook in a real Databricks workspace, replace placeholder paths such as `<storage-account>` and confirm that the Unity Catalog schemas exist.
All outputs have been cleared before publishing.


In [ ]:
from pyspark.sql.functions import count, sum as _sum, avg, lit, date_format, col

# =========================================================
# 1. READ SOURCE TABLES
# =========================================================
silver_clean_df = spark.table("corporate_travel_analytics.silver.bookings_clean")
silver_quarantine_df = spark.table("corporate_travel_analytics.silver.bookings_quarantine")
gold_active_detail_df = spark.table("corporate_travel_analytics.gold.booking_cost_detail")
gold_cancelled_detail_df = spark.table("corporate_travel_analytics.gold.cancelled_booking_detail")

# =========================================================
# 2. COUNTS
# =========================================================
active_count = gold_active_detail_df.count()
cancelled_count = gold_cancelled_detail_df.count()
quarantine_count = silver_quarantine_df.count()

# =========================================================
# 3. BOOKING SUMMARY
# =========================================================
booking_summary_df = gold_active_detail_df.agg(
    count("*").alias("total_valid_bookings"),
    _sum("flight_cost").alias("total_flight_cost"),
    _sum("hotel_cost").alias("total_hotel_cost"),
    _sum("total_cost").alias("total_booking_cost"),
    avg("total_cost").alias("avg_booking_cost")
).withColumn("total_quarantined_bookings", lit(quarantine_count))

booking_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("corporate_travel_analytics.gold.booking_summary")

# =========================================================
# 4. BOOKING SPEND BY DEPARTMENT
# =========================================================
booking_spend_by_department_df = (
    gold_active_detail_df
    .groupBy("department")
    .agg(
        count("*").alias("booking_count"),
        _sum("flight_cost").alias("total_flight_cost"),
        _sum("hotel_cost").alias("total_hotel_cost"),
        _sum("total_cost").alias("total_booking_cost"),
        avg("total_cost").alias("avg_booking_cost")
    )
)

booking_spend_by_department_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("corporate_travel_analytics.gold.booking_spend_by_department")

# =========================================================
# 5. BOOKING SPEND BY DESTINATION
# =========================================================
booking_spend_by_destination_df = (
    gold_active_detail_df
    .groupBy("destination")
    .agg(
        count("*").alias("booking_count"),
        _sum("flight_cost").alias("total_flight_cost"),
        _sum("hotel_cost").alias("total_hotel_cost"),
        _sum("total_cost").alias("total_booking_cost"),
        avg("total_cost").alias("avg_booking_cost")
    )
)

booking_spend_by_destination_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("corporate_travel_analytics.gold.booking_spend_by_destination")

# =========================================================
# 6. BOOKING MONTHLY TREND
# =========================================================
booking_monthly_trend_df = (
    gold_active_detail_df
    .groupBy("booking_month")
    .agg(
        count("*").alias("booking_count"),
        _sum("flight_cost").alias("total_flight_cost"),
        _sum("hotel_cost").alias("total_hotel_cost"),
        _sum("total_cost").alias("total_booking_cost"),
        avg("total_cost").alias("avg_booking_cost")
    )
    .orderBy("booking_month")
)

booking_monthly_trend_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("corporate_travel_analytics.gold.booking_monthly_trend")

# =========================================================
# 7. FAILED BOOKING SUMMARY
# =========================================================
failed_booking_summary_df = (
    silver_quarantine_df
    .groupBy("validation_error")
    .agg(
        count("*").alias("failed_record_count")
    )
    .orderBy(col("failed_record_count").desc())
)

failed_booking_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("corporate_travel_analytics.gold.failed_booking_summary")

# =========================================================
# 8. CANCELLED BOOKING SUMMARY
# =========================================================
cancelled_booking_summary_df = gold_cancelled_detail_df.agg(
    count("*").alias("total_cancelled_bookings"),
    _sum("flight_cost").alias("total_cancelled_flight_cost"),
    _sum("hotel_cost").alias("total_cancelled_hotel_cost"),
    _sum("total_cost").alias("total_cancelled_booking_cost"),
    avg("total_cost").alias("avg_cancelled_booking_cost")
)

cancelled_booking_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("corporate_travel_analytics.gold.cancelled_booking_summary")

print("Gold summary refresh completed successfully.")
